# Lab 3 · NumPy và tư duy vector hoá

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · Bài 3**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu bài lab**

1. Tính dung lượng và vị trí phần tử từ `dtype`, `shape`, `strides`.
2. Chọn dữ liệu bằng chỉ mục, lát cắt và mask; phân biệt view với bản sao.
3. Viết phép tính theo hàng, cột bằng broadcasting và phát hiện phép tính sai dù code vẫn chạy.
4. Đo thời gian, diễn giải thống kê và lấy mẫu để kiểm tra dữ liệu.

Dữ liệu được tự tạo để có thể tính tay. Các ví dụ giá chỗ ở là **giả lập**, đơn vị **USD/đêm**, không phải kết quả phân tích Inside Airbnb.

**BẢN PILOT — dùng thử private grader; chưa phải điểm chính thức của môn học.**
Phiên bản đề: `numpy-vectorization-v2` (20/09/2026). Lưu đúng tên **`lab-03.ipynb`**.

## Cách làm và nộp bài

- Lab ở chế độ **✅ mở**: được dùng AI; cần đọc hiểu, kiểm chứng và khai báo.
- Ghi dự đoán trước khi chạy; điền thân hàm TODO và phần trả lời Markdown.
- Giữ tên hàm, tham số, cell ID và metadata; viết helper/import thêm trong chính cell bài làm.
  Grader chỉ chạy cell câu đó với input riêng, không phụ thuộc biến A/B/gia ở cell khác.
- Public checks chỉ là ví dụ tự kiểm tra; test ẩn dùng dữ liệu khác cùng hợp đồng.
  Một check lỗi không dừng các check sau. TODO báo **CHƯA LÀM**; lỗi cú pháp vẫn cần sửa.
- Không sửa input tại chỗ; không ghi cứng kết quả. Không cần xử lý input ngoài hợp đồng.
- **Restart & Run all**, lưu notebook, commit/push vào repo của bạn trong org và tạo tag theo cuối bài.
  Không cần workflow/secret ở repo sinh viên. Notebook chạy hết khi còn TODO chưa có nghĩa đạt bài.

## Rubric dự kiến — 100 điểm

| Mục | Điểm | Cách đánh giá |
|---|---:|---|
| Q1–Q6, mỗi câu 10 điểm | 60 | Tự động; giảng viên kiểm tra phương pháp |
| Q8: thống kê và lấy mẫu | 20 | Tự động |
| Dự đoán, giải thích Q1–Q6 và diễn giải Q8 | 10 | Giảng viên |
| Q7: nhận xét phép đo thời gian | 5 | Giảng viên, không chấm tỷ lệ tốc độ cố định |
| Khai báo AI và bằng chứng kiểm chứng | 5 | Giảng viên |

Q7 giữ code đo thời gian để quan sát; không phải hàm TODO tính điểm tự động.
Yêu cầu slicing/np.ix_/vector hóa được giảng viên kiểm tra cùng code, không suy ra chỉ từ kết quả.

In [ ]:
# Public checks giúp tự kiểm tra; đây không phải điểm chính thức.
# Không sửa cell này trong bài nộp.
PUBLIC_RESULTS = {}

def public_check(case_id, check):
    try:
        check()
    except NotImplementedError:
        status, detail = "CHƯA LÀM", "Điền phần TODO rồi chạy lại."
    except AssertionError as exc:
        status, detail = "CHƯA ĐẠT", str(exc) or "Kết quả chưa khớp ví dụ công khai."
    except Exception as exc:
        status, detail = "LỖI", f"{type(exc).__name__}: {exc}"
    else:
        status, detail = "ĐẠT", "Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric."
    PUBLIC_RESULTS[case_id] = status
    print(f"[{status}] {case_id}: {detail}")

def require_answer(value):
    if value is None or value is Ellipsis:
        raise NotImplementedError

def preview(label, action):
    try:
        value = action()
    except NotImplementedError:
        print(f"{label}: chưa chạy được vì còn TODO.")
    except Exception as exc:
        print(f"{label}: {type(exc).__name__}: {exc}")
    else:
        print(label)
        print(value)

def show_public_summary():
    print("PUBLIC CHECKS — không phải điểm chính thức")
    for case_id, status in PUBLIC_RESULTS.items():
        print(f"{case_id}: {status}")
    print("Sau khi sửa, Restart & Run all để làm mới toàn bộ kết quả.")


## Dữ liệu dùng trong lab

`A` là mảng 4 hàng × 3 cột. Các bài có sửa dữ liệu sẽ làm trên bản sao của `A`.

In [ ]:
import numpy as np
from timeit import repeat

A = np.array([[10, 12, 11],
              [20, 21, 24],
              [30, 33, 31],
              [40, 44, 42]], dtype=np.int64)
print(A)

## Bài 1 · Đọc cấu trúc mảng

1. Ghi `shape`, `itemsize`, `nbytes` và `strides` của `A` trước khi chạy code.
2. `A[3, 1]` cách phần tử đầu bao nhiêu byte?
3. Nếu đổi sang `int32`, dung lượng dữ liệu và strides thay đổi thế nào?

`itemsize`: số byte mỗi phần tử. `nbytes`: dung lượng dữ liệu của mảng, chưa gồm thông tin quản lý đối tượng.

### Hợp đồng hàm · 10 điểm tự động

`array_layout(matrix: np.ndarray, row: int, col: int) -> dict`

Nhận ndarray số nguyên 2 chiều, C-contiguous, ít nhất một hàng và cột;
`row`, `col` là chỉ số hợp lệ không âm. Trả dict có đúng các khóa:
`shape`, `itemsize`, `nbytes`, `strides`, `offset`, `int32_array`.
`shape` và `strides` là tuple; `offset` tính bằng byte từ đầu dữ liệu tới `[row, col]`.
`int32_array` là ndarray chứa cùng giá trị, dtype `int32`; các số đầu vào nằm trong miền int32.
Không ghi cứng shape, dtype hoặc chỉ số của A. Thực hành bằng `array_layout(A, 3, 1)`.

**Dự đoán và giải thích:**

- A: …
- Vị trí A[3, 1]: …
- Khi đổi sang int32: …

In [ ]:
def array_layout(matrix: np.ndarray, row: int, col: int) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q1():
    result = array_layout(A, 3, 1)
    assert set(result) == {"shape", "itemsize", "nbytes", "strides", "offset", "int32_array"}
    assert result["shape"] == (4, 3) and result["itemsize"] == 8 and result["nbytes"] == 96
    assert result["strides"] == (24, 8) and result["offset"] == 80
    assert result["int32_array"].dtype == np.dtype("int32")
    np.testing.assert_array_equal(result["int32_array"], A)

public_check("Q1 · ví dụ công khai", check_q1)

## Bài 2 · Lát cắt và view

Tạo `V` gồm hàng 1, 3 và cột 0, 2 của `A`, **bằng một lát cắt cho mỗi chiều**.

1. Dự đoán giá trị, shape và strides của `V`.
2. `V[1, 1]` trỏ tới ô nào trong `A`? Ô đó cách đầu dữ liệu của `A` bao nhiêu byte?
3. Kiểm tra `V` có dùng chung dữ liệu với `A` không.

Dùng `np.shares_memory(A, V)` để kiểm tra việc dùng chung bộ nhớ.

### Hợp đồng hàm · 10 điểm tự động

`slice_view(matrix: np.ndarray) -> dict`

Nhận ndarray số nguyên 2 chiều, C-contiguous, ít nhất 4 hàng và 3 cột.
Lấy hàng bắt đầu từ 1, bước 2 và cột bắt đầu từ 0, bước 2 **bằng slicing**.
Trả dict `view`, `shape`, `strides`, `offset_v11`, `shares_memory`.
`view` phải thực sự chia sẻ bộ nhớ với input; không chỉ trả boolean `True`.
`shape`, `strides` là tuple của view; `offset_v11` tính từ đầu **matrix** đến `view[1, 1]`,
không phải từ đầu view. Không sửa giá trị trong view. Không giả định input luôn 4 × 3.

**Dự đoán:** giá trị …; shape …; strides …; ô trong A …; độ lệch …

In [ ]:
def slice_view(matrix: np.ndarray) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q2():
    result = slice_view(A)
    assert set(result) == {"view", "shape", "strides", "offset_v11", "shares_memory"}
    np.testing.assert_array_equal(result["view"], [[20, 24], [40, 42]])
    assert result["shape"] == (2, 2) and result["strides"] == (48, 16)
    assert result["offset_v11"] == 88 and result["shares_memory"]
    assert np.shares_memory(A, result["view"])

public_check("Q2 · ví dụ công khai", check_q2)

## Bài 3 · View và bản sao

Đọc code dưới và dự đoán `B[0]` sau **mỗi** lệnh gán. Sau đó chạy để kiểm tra.

### Hợp đồng hàm · 10 điểm tự động

`copy_columns(matrix: np.ndarray) -> np.ndarray`

Nhận ndarray số hữu hạn 2 chiều, ít nhất một hàng và ba cột.
Trả ndarray gồm cột 1 và 2, giữ nguyên dtype và thứ tự hàng.
Kết quả không chia sẻ bộ nhớ với input; sửa kết quả không làm đổi input.
Hàm phải dùng tham số `matrix`, không phụ thuộc biến B của ví dụ. Thử `copy_columns(B)`.

**Dự đoán:**

- Sau `C[0, 0] = -9`: B[0] = …
- Sau `V[0, 0] = -1`: B[0] = …
- Giải thích sự khác biệt: …

In [ ]:
B = A.copy()
V = B[:, 1:3]
C = B[:, [1, 2]]
C[0, 0] = -9
print("Sau khi sửa C:", B[0])
V[0, 0] = -1
print("Sau khi sửa V:", B[0])

Tạo `D` chứa cột 1 và 2 của `B` sao cho sửa `D` không làm đổi `B`.

In [ ]:
def copy_columns(matrix: np.ndarray) -> np.ndarray:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q3():
    before = B.copy()
    result = copy_columns(B)
    np.testing.assert_array_equal(result, B[:, 1:3])
    assert result.dtype == B.dtype and not np.shares_memory(result, B)
    result[0, 0] = 999
    np.testing.assert_array_equal(B, before)
    np.testing.assert_array_equal(A[0], [10, 12, 11])

public_check("Q3 · ví dụ công khai", check_q3)

## Bài 4 · Chọn phần tử

Không dùng vòng `for`:

1. Lấy các số chẵn trong `A` theo thứ tự từng hàng.
2. Lấy hai ô `A[0, 1]` và `A[2, 2]` bằng hai mảng chỉ mục.
3. Lấy cột 1, 2 ở mỗi hàng 0, 2 bằng `np.ix_`.

Ghi shape của ba kết quả và giải thích vì sao chúng khác nhau.

### Hợp đồng hàm · 10 điểm tự động

`select_elements(matrix: np.ndarray) -> dict`

Nhận ndarray số nguyên 2 chiều, ít nhất 3 hàng và 3 cột. Trả dict:
`so_chan`: ndarray 1D các số chẵn theo thứ tự hàng (có thể rỗng);
`hai_o`: ndarray gồm `[matrix[0, 1], matrix[2, 2]]`;
`bon_o`: ndarray 2 × 2 tại hàng `[0, 2]`, cột `[1, 2]`.
Dùng mask, hai mảng chỉ mục và `np.ix_` tương ứng, không dùng vòng lặp/comprehension.
Cả ba kết quả giữ dtype và không chia sẻ bộ nhớ với input.

In [ ]:
def select_elements(matrix: np.ndarray) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q4():
    result = select_elements(A)
    assert set(result) == {"so_chan", "hai_o", "bon_o"}
    np.testing.assert_array_equal(result["so_chan"], [10, 12, 20, 24, 30, 40, 44, 42])
    np.testing.assert_array_equal(result["hai_o"], [12, 31])
    np.testing.assert_array_equal(result["bon_o"], [[12, 11], [33, 31]])
    for value in result.values():
        assert isinstance(value, np.ndarray) and value.dtype == A.dtype
        assert not np.shares_memory(A, value)

public_check("Q4 · ví dụ công khai", check_q4)

**Giải thích shape:** …

## Bài 5 · Broadcasting

Tạo `K` qua hai bước:

1. Từ `A`, cộng 1, 2, 3 vào lần lượt các cột 0, 1, 2.
2. Trên kết quả vừa tính, cộng 10, 20, 30, 40 vào lần lượt các hàng 0, 1, 2, 3.

Ví dụ: `K[1, 2] = 24 + 3 + 20 = 47`.

Viết bằng NumPy, không dùng `for`. Ghi shape của từng mảng dùng để cộng.

### Hợp đồng hàm · 10 điểm tự động

`broadcast_add(matrix: np.ndarray, column_add: np.ndarray, row_add: np.ndarray) -> np.ndarray`

Nhận ma trận số hữu hạn 2 chiều không rỗng; `column_add` là ndarray 1D dài bằng
số cột, `row_add` là ndarray 1D dài bằng số hàng. Trả ndarray cùng shape với ma trận,
mỗi ô `[i, j]` bằng `matrix[i, j] + column_add[j] + row_add[i]`.
Hỗ trợ số nguyên và số thực; dùng broadcasting, không dùng vòng lặp/comprehension.
Không sửa bất kỳ input nào. Không giả định số hàng bằng số cột.

In [ ]:
b = np.array([1, 2, 3])
d = np.array([10, 20, 30, 40])

In [ ]:
def broadcast_add(matrix: np.ndarray, column_add: np.ndarray, row_add: np.ndarray) -> np.ndarray:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q5():
    before = A.copy()
    result = broadcast_add(A, np.array([1, 2, 3]), np.array([10, 20, 30, 40]))
    assert isinstance(result, np.ndarray) and result.shape == A.shape
    np.testing.assert_array_equal(result, [[21, 24, 24], [41, 43, 47], [61, 65, 64], [81, 86, 85]])
    np.testing.assert_array_equal(A, before)

public_check("Q5 · ví dụ công khai", check_q5)

**Giải thích:** shape của b …; của d …; shape dùng khi cộng theo hàng …; vì sao `A + d` lỗi …

## Bài 6 · Sửa phép tính theo hàng

Mỗi hàng dưới đây là giá của một chỗ ở qua 3 kỳ thu thập (**giả lập, USD/đêm**).
Mỗi giá cần trừ đi trung bình của chính hàng đó.

Ví dụ: `[40, 50, 60]` → `[-10, 0, 10]`.

Code chạy được nhưng tính sai. Hãy sửa và giải thích lỗi.

### Hợp đồng hàm · 10 điểm tự động

`center_rows(prices: np.ndarray) -> tuple`

Nhận ndarray số hữu hạn 2 chiều không rỗng, số nguyên hoặc số thực.
Trả tuple `(trung_binh_hang, chenh_lech)` gồm hai ndarray float:
mean của từng hàng có shape `(số_hàng, 1)` và độ lệch có shape giống input.
Dùng `axis` và broadcasting, không dùng vòng lặp/comprehension.
Không giả định input là ma trận vuông. Với hàng một phần tử, độ lệch bằng 0.

In [ ]:
gia = np.array([[40, 50, 60],
                [60, 60, 60],
                [30, 60, 90]], dtype=np.float64)
tb_sai = gia.mean(axis=1)
sai = gia - tb_sai
print(sai)

In [ ]:
def center_rows(prices: np.ndarray) -> tuple:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q6():
    means, deviations = center_rows(gia)
    assert means.shape == (3, 1) and deviations.shape == gia.shape
    assert np.issubdtype(means.dtype, np.floating) and np.issubdtype(deviations.dtype, np.floating)
    np.testing.assert_allclose(means, [[50], [60], [60]])
    np.testing.assert_allclose(deviations, [[-10, 0, 10], [0, 0, 0], [-30, 0, 30]])
    np.testing.assert_allclose(deviations.mean(axis=1), 0, atol=1e-12)

public_check("Q6 · ví dụ công khai", check_q6)

**Giải thích:** `tb_sai` có shape … nên bị ghép theo …; cách sửa …

## Bài 7 · Đo thời gian

Ba cách dưới tính cùng phép nhân 2. Chạy phép đo với mảng nhỏ và mảng lớn, rồi trả lời:

1. Cách nào vẫn lặp trong Python?
2. Chuyển list sang ndarray có đủ để làm vòng `for` nhanh hơn không?
3. Kết quả đo có thay đổi theo số phần tử không? Vì sao?

Dữ liệu được tạo trước khi đo. Ba cách đều tạo kết quả mới; phép đo dùng thời gian nhỏ nhất trong 3 lần để giảm ảnh hưởng của nhiễu. Không yêu cầu một tỷ lệ nhanh/chậm cố định.

**Q7 — 5 điểm giảng viên:** ghi số đo thực tế và giải thích; không yêu cầu máy nào cũng nhanh hơn theo cùng một tỷ lệ.

In [ ]:
def nhan_list(xs):
    return [v * 2 for v in xs]

def nhan_for_array(x):
    return [v * 2 for v in x]

def nhan_array(x):
    return x * 2

for n in (10, 100_000):
    x = np.arange(n, dtype=np.int64)
    xs = x.tolist()
    np.testing.assert_array_equal(nhan_list(xs), nhan_array(x))
    np.testing.assert_array_equal(nhan_for_array(x), nhan_array(x))
    so_lan = 100 if n == 10 else 5
    print(f"\nn = {n:,}")
    for ten, ham, dau_vao in [("for trên list", nhan_list, xs),
                             ("for trên ndarray", nhan_for_array, x),
                             ("phép toán mảng", nhan_array, x)]:
        t = min(repeat(lambda: ham(dau_vao), number=so_lan, repeat=3)) / so_lan
        print(f"{ten}: {t * 1e6:.2f} µs/lần")

**Nhận xét từ phép đo của bạn:** …

**Giải thích bằng cách thực thi và cách lưu dữ liệu:** …

## Bài 8 · Thống kê giá và lấy mẫu

Giá của 5 chỗ ở (**giả lập, USD/đêm**): `[40, 50, 60, 70, 380]`.

1. Tính trung bình, trung vị, độ lệch chuẩn bằng NumPy.
2. Tính tỷ lệ chỗ ở có giá không quá 70 USD.
3. Lấy ngẫu nhiên 3 dòng, không lặp lại dòng, với seed 42. Tính trung bình mẫu.

Dùng `rng.choice(len(gia_dem), size=3, replace=False)` để lấy chỉ mục. `replace=False` không chọn trùng một dòng.

### Hợp đồng hàm · 20 điểm tự động

`price_statistics(prices: np.ndarray, threshold: float = 70.0, sample_size: int = 3, seed: int = 42) -> dict`

Nhận ndarray float 1D không rỗng, các giá hữu hạn không âm;
`threshold` hữu hạn, `1 <= sample_size <= prices.size`, `seed` là số nguyên không âm.
Trả dict có đúng các khóa:
- `mean`, `median`, `std`: float, độ lệch chuẩn **tổng thể** `ddof=0`.
- `share`: float, tỷ lệ giá **<= threshold**, thang 0–1.
- `indices`: ndarray số nguyên 1D, dài `sample_size`, thứ tự giữ nguyên từ lời gọi
  `rng = np.random.default_rng(seed)` rồi **một lần**
  `rng.choice(len(prices), size=sample_size, replace=False)`.
- `sample`: ndarray `prices[indices]`, giữ thứ tự chỉ mục; `sample_mean`: float.

Không làm tròn, không sửa input hoặc RNG toàn cục. Seed giống nhau phải cho kết quả giống nhau.
Giá trùng nhau vẫn có thể xuất hiện trong mẫu nếu thuộc các dòng khác nhau.
Thực hành với `[40, 50, 60, 70, 380]` và các tham số mặc định.

In [ ]:
gia_dem = np.array([40, 50, 60, 70, 380], dtype=np.float64)

In [ ]:
def price_statistics(prices: np.ndarray, threshold: float = 70.0, sample_size: int = 3, seed: int = 42) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q8():
    prices = np.array([40, 50, 60, 70, 380], dtype=np.float64)
    result = price_statistics(prices)
    assert set(result) == {"mean", "median", "std", "share", "indices", "sample", "sample_mean"}
    assert result["mean"] == 120 and result["median"] == 60 and result["share"] == 0.8
    np.testing.assert_allclose(result["std"], np.sqrt(17000))
    indices = result["indices"]
    assert indices.shape == (3,) and np.issubdtype(indices.dtype, np.integer)
    np.testing.assert_array_equal(indices, np.random.default_rng(42).choice(5, size=3, replace=False))
    np.testing.assert_array_equal(result["sample"], prices[indices])
    assert np.isclose(result["sample_mean"], np.mean(prices[indices]))

public_check("Q8 · ví dụ công khai", check_q8)

**Trả lời ngắn:**

- Trong báo cáo bài tập lớn, bạn dùng trung bình hay trung vị để mô tả mức giá điển hình của dãy này? Vì sao? …
- Có nên xoá giá 380 chỉ vì nó cao hơn các giá còn lại không? …
- Trung bình mẫu có bằng trung bình cả 5 giá không? Seed cố định có bảo đảm mẫu đại diện không? …
- Khi đã có toàn bộ dữ liệu hợp lệ, lấy mẫu có cần thiết để tính giá trung bình không? Khi nào việc đọc một mẫu dòng vẫn hữu ích? …

## Khai báo sử dụng AI

- **Công cụ đã dùng:** … *(hoặc “không dùng”)*
- **Bài đã dùng AI và yêu cầu chính:** …
- **Một gợi ý đã kiểm chứng, cách kiểm chứng và kết quả:** …

Không chỉ ghi “code chạy được”; nêu phép tính tay, shape hoặc ví dụ đối chiếu đã dùng.

---

## Tóm tắt bài lab

| Nội dung chính | Cần tự giải thích được |
|---|---|
| Shape, dtype, strides | Dung lượng dữ liệu và vị trí một phần tử |
| Lát cắt, view và bản sao | Sửa kết quả có làm đổi mảng gốc không |
| Mask và mảng chỉ mục | Những ô nào được chọn, shape kết quả |
| Broadcasting và axis | Mỗi phần tử được tính với giá trị nào |
| Đo thời gian | Vì sao phép toán trên mảng khác vòng for Python |
| Thống kê và lấy mẫu | Con số có phù hợp với câu hỏi trong bài tập lớn không |

**Trước khi nộp:** hoàn thành TODO và câu trả lời, chạy Restart & Run all, lưu notebook.

**Bài sau:** pandas — làm việc với bảng có tên cột và nhiều kiểu dữ liệu.

## Thông tin và nộp bài

**Họ tên / MSSV / lớp:** [Điền]

1. Giữ tên file **`lab-03.ipynb`** ở gốc repo (dấu gạch ngang, không phải `lab_03.ipynb`).
2. Hoàn thành hàm TODO, dự đoán, nhận xét Q7, diễn giải Q8 và khai báo AI.
3. Restart & Run all; đọc bảng public checks, lưu file rồi commit/push lên `main`.
4. Tạo tag `submit/lab-03/v1`; lần sửa tiếp dùng v2 hoặc v3, không di chuyển tag đã nộp.
5. Repo cần được giảng viên đăng ký chấm. Xem status `private-grader/lab-03` trên commit nộp;
   điểm tự động tối đa 80, còn 20 điểm chờ giảng viên. Xem lịch quét trong README.

Tag v1/v2/v3 là **lần nộp**, không phải phiên bản đề. Không đổi metadata để chọn bộ test khác.

In [ ]:
show_public_summary()